In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI
from DatabaseUtils import PostgresQueryExecutor
from geopy.distance import geodesic
import json
import os
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
custom_base_url = "Change your url here"
model_name = 'Change your url here'
api_key = 'Change your api key here'
llm = ChatOpenAI(model_name=model_name, openai_api_key=api_key, base_url=custom_base_url)

In [ ]:
json_file = '../dataset/train.json'
with open(json_file, 'r', encoding='utf-8') as f:
    datas = json.load(f)
all_intent  = []
for data in datas:
    intent = data['intent']
    if intent not in all_intent:
        all_intent.append(intent)

In [ ]:
from rank_bm25 import BM25Okapi
import jieba
corpus = all_intent
tokenized_corpus = [list(jieba.cut_for_search(doc)) for doc in corpus]
# 构建索引
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
template_path = './other_prompts/ICL_prompt.txt'
with open(template_path, "r", encoding="utf-8") as file:
    template = file.read()

In [ ]:
import ast
import re

def extract_intent_and_slots(text):
    # Regular expression patterns to extract intent and slots
    intent_pattern = r"intent[:>)]?\s*(.*?)(?=[<\(]_?slots_?[>\)])"
    slots_pattern = r"[<\(]_?slots_?[>\)]\s*(\[.*?\])"
    #intent_pattern = r"intent[:>)]?\s*([^,]+?)(?=[<\(]_?slots_?[:>)]?)"
    #slots_pattern = r"[<\(]_?slots_?[:>)]?\s*(\[.*?\])"

    # Extracting intent
    intent_match = re.search(intent_pattern, text)
    if intent_match:
        intent = intent_match.group(1).strip()
    else:
        intent = None

    # Extracting slots
    slots_match = re.search(slots_pattern, text)
    if slots_match:
        slots_str = slots_match.group(1)
        try:
            slots_list = ast.literal_eval(slots_str)  # Safely evaluates the string as a list
            for i in range(len(slots_list)):
                if "poi" in slots_list[i]:
                    slots_list[i] = slots_list[i].replace("poi","POI")
        except (SyntaxError, ValueError) as e:
            print(f"Error parsing slots: {e}")
            slots_list = None
    else:
        slots_list = None

    return intent, slots_list

def slots_to_dict(slots_list):
    slots_dict = {}
    if slots_list:
        for slot in slots_list:
            key, value = slot.split(':')
            slots_dict[key.strip()] = value.strip()
    return slots_dict



def extract_slu(text: dict):
    pred = text.lower().strip()
    pred = pred.replace('_', '')
    pred = pred.replace('\n', '')
    pred = pred.replace('输出:','')
    if 'intent:' in pred:
        pred = pred.replace('intent:', '<intent>')
    if 'slots:' in pred:
        pred = pred.replace('slots:', '<slots>')
    if ' slots ' in pred:
        pred = pred.replace(' slots ', '<slots>')
    if '<<slots>>' in pred:
        pred = pred.replace('<<slots>>', '<slots>')
    if '」' in pred: 
        pred = pred.replace('」', '\']')
    try:
        intent, slots = extract_intent_and_slots(pred)
        intent = bm25.get_top_n(list(jieba.cut_for_search(intent)), corpus, n=1)
        if intent != None and slots != None:
            extract_intent = intent
            extract__slots = slots
        else:
            print(f'This：{pred}')
            extract_intent = 'error'
            extract__slots = text
    except:
        print(f'This：{pred}')
        extract_intent = 'error'
        extract__slots = text
    return extract_intent, extract__slots

In [ ]:
def restore_keywords_from_query(query, token_slot):
    keywords = []
    current_tokens = []
    current_label = None
    query = list(query)
    token_slot = token_slot[1:-1]

    for token, slot in zip(query, token_slot):
        if slot.startswith('B-'):
            if current_tokens:
                keywords.append((''.join(current_tokens), current_label))
                current_tokens = []
            current_label = slot[2:]
            current_tokens.append(token)
        elif slot.startswith('I-') and current_label == slot[2:]:
            current_tokens.append(token)
        else:
            if current_tokens:
                keywords.append((''.join(current_tokens), current_label))
                current_tokens = []
                current_label = None

    if current_tokens:
        keywords.append((''.join(current_tokens), current_label))
    keyword_pair = []
    for keyword in keywords:
        if keyword[-1] == 'city':
            keyword_pair.append(f'城市:{keyword[0]}')
        elif keyword[-1] == 'district':
            keyword_pair.append(f'区域名称:{keyword[0]}')
        elif keyword[-1] == 'POI':
            keyword_pair.append(f'POI名称:{keyword[0]}')
        elif keyword[-1] == 'community':
            keyword_pair.append(f'小区名称:{keyword[0]}')
        elif keyword[-1] in ['POI:shopping_mall', 'POI:supermarket', 'POI:hospital',
                            'POI:park', 'POI:subway_station', 'POI:bus_stop',
                            'POI:primary_school', 'POI:middle_school']:
            keyword_pair.append(f'POI类型:{keyword[0]}')
        elif keyword[-1] == 'POI_third':
            keyword_pair.append(f'POI类型标签:{keyword[0]}')
        elif keyword[-1] == 'sale_status':
            keyword_pair.append(f'销售状态:{keyword[0]}')
        elif keyword[-1] == 'community_tag':
            keyword_pair.append(f'小区属性:{keyword[0]}')
        elif keyword[-1] == 'distance':
            keyword_pair.append(f'距离:{keyword[0]}')
        elif keyword[-1] == 'house_price':
            keyword_pair.append(f'成交均价:{keyword[0]}')
        elif keyword[-1] == 'green_rate':
            keyword_pair.append(f'绿化率:{keyword[0]}')

    return keyword_pair

In [ ]:
from tqdm import tqdm
json_file = '../dataset/test.json'
with open(json_file, 'r', encoding='utf-8') as f:
    datas = json.load(f)

for data in tqdm(datas):
    query = data['query']
    slots = data["tokens_slots(bert)"]
    data['true_slots_name'] = restore_keywords_from_query(query, slots)
    prompt = ChatPromptTemplate.from_template(template)
    chain = prompt | llm
    key_words = restore_keywords_from_query(query, slots)
    response = chain.invoke({"question":query})
    response = re.sub(r'<think>.*?</think>', '', response.content, flags=re.S)
    # print(response.content)
    data['ICL_pred_intent'], data['ICL_pred_slots'] = extract_slu(response)

save_json_file = '../results/test-with-ICL_pred_intent+slots.json'
with open(save_json_file, 'w', encoding='utf-8') as file:
    json.dump(datas, file, ensure_ascii=False, indent=4)

In [ ]:
def metric_compute(trues: list, preds: list):
    if len(trues) != len(preds):
        return 'Input lengthes not equal!'
    precision = 0
    precision_all = 0
    recall = 0
    recall_all = 0
    for true_label, pred_label in zip(trues, preds):
        if isinstance(true_label, type('')):
            true_label = [true_label]
        if isinstance(pred_label, type('')):
            pred_label = [pred_label]
        for pred in pred_label:
            if pred in true_label:
                precision += 1
            precision_all += 1
        for true in true_label:
            if true in pred_label:
                recall += 1
            recall_all += 1
    P = precision/precision_all
    R = recall/recall_all
    F1 = 2 * P * R / (P + R)
    return P, R, F1

In [ ]:
datas_saved_file = '../results/test-with-ICL_pred_intent+slots.json'
with open(datas_saved_file, 'r', encoding='utf-8') as f:
    datas = json.load(f)
predicts = []
trues = []

for data in datas:
    trues.append(data['intent'])
    try:
        predicts.append(data['ICL_pred_intent'])
    except:
        print(data)

print('ICL Intent prediction results')
metric_compute(trues, predicts)

In [ ]:
datas_saved_file = '../results/test-with-ICL_pred_intent+slots.json'
with open(datas_saved_file, 'r', encoding='utf-8') as f:
    datas = json.load(f)
predicts = []
trues = []

for data in datas:
    trues.append(data['true_slots_name'])
    try:
        predicts.append(data['ICL_pred_slots'])
    except:
        print(data)

print('ICL Slots prediction results')
metric_compute(trues, predicts)

In [ ]:
# read templates
prompt_file_name = './other_prompts/retrival_prompt.json'
with open(prompt_file_name, 'r', encoding='utf-8') as f:
    templates = json.load(f)

In [ ]:
from rank_bm25 import BM25Okapi
import jieba

# Read the table names and store them as a vector in memory.
table_names_file = './table_names.json'
with open(table_names_file, 'r', encoding='utf-8') as f:
    name_data = json.load(f)
# Corpus
corpus = name_data
tokenized_corpus = [list(jieba.cut_for_search(doc)) for doc in corpus]
# Build the index
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
def get_table_name(query, intent, slots): 
    prompt = ChatPromptTemplate.from_template(templates[intent])
    chain = prompt | llm
    response = chain.invoke({"question":query,"intent":intent, "slots":slots}).content
    # Perform preliminary cleaning on content generated by the LLM
    if '：' in response or ':' in response:
        #print(data['predict_tabel_name'])
        if '：' in response:
            idx = response.rfind('：')
        if ':' in response:
            idx = response.rfind(':')
        response = response[idx+1:]
    if ',' in response and '[' in response:
        #print(data['predict_tabel_name'])
        #print('%%%%%%%%%%')
        predict = response.split(',')
        for pred in predict:
            pred = pred.replace("[", "")
            pred = pred.replace("]", "")
            pred = pred.replace("'", "")
            pred = pred.replace('"', '')
    else:
        predict = response
        predict = predict.strip("[]'\"")
        if ',' in predict:
            predict = predict.split(',')
    # Remove the spaces
    if isinstance(predict, type([])):
        bm_25 = predict.copy()
        for i,pred in enumerate(predict):
            pred = pred.replace(" ", "")
            pred = pred.replace("[", "")
            pred = pred.replace("]", "")
            pred = pred.replace('\\', "")
            pred = pred.replace("\"", "")
            pred = pred.replace("\'", "")
            pred = pred.replace("\n", "")
            predict[i] = pred
            # result = retriever.invoke(pred)
            # Retrieve BM25 matching results
            tokenized_query = list(jieba.cut_for_search(pred))
            result = bm25.get_top_n(tokenized_query, corpus, n=1)
            # result = [doc.page_content for doc in result]
            bm_25[i] = result[0]
    if isinstance(predict, type([])):
        predict = list(set(predict))
        bm_25 = list(set(bm_25))
            
    if isinstance(predict, type('')):
        pred = predict
        pred = pred.replace(" ", "")
        pred = pred.replace("[", "")
        pred = pred.replace("]", "")
        pred = pred.replace('\\', "")
        pred = pred.replace("\"", "")
        pred = pred.replace("\'", "")
        pred = pred.replace("\n", "")
        predict = pred
        #result = retriever.invoke(pred)
        tokenized_query = list(jieba.cut_for_search(pred))
        result = bm25.get_top_n(tokenized_query, corpus, n=1)
        # result = [doc.page_content for doc in result]
        bm_25 = result[0]
    
    return predict

In [ ]:
Database_agent = create_react_agent(
    model=llm,
    prompt=(
        '''
        你是一个负责数据库SQL语句生成的代理，你的主要任务是根据用户的问题生成对应的查询语句
        以下是你工作的主要内容：
        - 你只需要对用户的问题进行查询语句的生成，无需回答用户的问题
        - 对于最值类的问题，除非用户指定数量，否则你需要将最终的结果数量限制为1
        - 你需要按照用户的要求对答案进行规范化输出，输出内容中不要包含除SQL语句外任何无关的内容
        - 在生成的查询语句中，你需要根据用户的问题找到最相关的属性列的形成查询语句，你可以按照用户的需求对返回的属性列进行排序
        - 你永远只能生成SELECT操作的SQL语句，SQL语句中不要出现任何可能会破坏数据库中数据的操作，包括但不限于INSERT, UPDATE, DELETE, DROP等
        - 对于用户的问题，你只能生成一条SQL语句来完成查询
        - 在回答完问题后，需要将答案交给中枢系统，你无需做任何处理
        '''
    ),
    tools=[],
    name="Database_agent",
)

In [ ]:
def get_distance(origin, destination, type):
    """    
    根据起始地点和终点的经纬度，计算两个地点之间指定类型的距离。 
           
    参数:    
    origin (list): 起始地点的经纬度，经度在前，纬度在后。    
    destination (list): 终点的经纬度，经度在前，纬度在后。       
    type (int)：计算距离的方式，直线距离为0，车行距离为1，步行距离为3。

    返回:    
    float: 两个地点之间类型的距离（单位：公里）。    
    """    

    type_list = ["直线距离","车行距离","","步行距离"]
    if type == 0:
        origin = (origin[1], origin[0])
        destination = (destination[1], destination[0])
        return round(geodesic(origin, destination).kilometers, 2)
    else:
        sql = f'''
        SELECT "距离" FROM "{type_list[type]}表" 
        WHERE "起点经纬度" = '{origin[0]},{origin[1]}' AND 
        "终点经纬度" = '{destination[0]},{destination[1]}';
        '''
    try:
        executor = PostgresQueryExecutor()
        _, distance = executor.execute_sql(sql)
        executor.close()
        return distance[0][0]
    except:
        return None    

In [ ]:
def get_time(origin, destination, transport):
    """    
    根据起始地点和终点的经纬度，计算两个地点之间指定交通方式所需要的时间。 
           
    参数:    
    origin (list): 起始地点的经纬度，经度在前，纬度在后。    
    destination (list): 终点的经纬度，经度在前，纬度在后。       
    transport (str)：交通方式类型，可选参数范围为步行，开车，骑车，公共交通。

    返回:    
    int: 两个地点之间指定交通方式所需要的时间（单位：分钟）。    
    """ 
    if transport in ["步行", "开车", "骑车", "公共交通"]:
        sql = f'''
        SELECT "时间" FROM "{transport}时间表" 
        WHERE "起点经纬度" = '{origin[0]},{origin[1]}' AND 
        "终点经纬度" = '{destination[0]},{destination[1]}';
        '''
    else:
        return None
    
    try:
        executor = PostgresQueryExecutor()
        _, time = executor.execute_sql(sql)
        executor.close()
        return time[0][0]
    except:
        return None

In [ ]:
def get_poi(location, city, radius, types):
    """    
    根据中心点的经纬度，计算其周围指定半径范围内所有指定类型的POI的名称，POI与中心点的距离，和POI的数量。 
           
    参数:    
    location (list): 中心点的经纬度，经度在前，纬度在后。    
    city (str): 中心点所在的城市，格式为某某市。       
    radius (int)：指定的半径大小。
    types (str)：POI的类型（如景区），有些POI需要带上特定的标签（如5A景区）

    返回:    
    int: 中心点周围指定半径范围内所有指定类型的POI的数量。
    list: 中心点周围指定半径范围内所有指定类型的POI的名称和距离中心点的距离，列表中每一个元素中第一维是POI名称，第二维是距离中心点的距离    
    """ 

    if "三甲" in types or "重点" in types:
        sql = f'''
        SELECT "POI名称", "距离" FROM "小区生活圈表"
        WHERE "小区经纬度" = '{location[0]},{location[1]}' AND 
        "城市名" = '{city}' AND "POI二级标签" = '{types[2:]}' AND 
        "POI三级标签" = '{types[0:2]}' AND "距离" <= '{radius}';
        '''
    else:
        sql = f'''
        SELECT "POI名称", "距离" FROM "小区生活圈表"
        WHERE "小区经纬度" = '{location[0]},{location[1]}' AND 
        "城市名" = '{city}' AND "POI二级标签" = '{types}' AND "距离" <= '{radius}';
        '''
    
    try:
        executor = PostgresQueryExecutor()
        _, pois = executor.execute_sql(sql)
        executor.close()
        return len(pois), pois
    except:
        return None

In [ ]:
def get_rush_hours(origin, destination, transport, time_point):
    """    
    根据起始地点和终点的经纬度，计算两个地点之间指定交通方式在指定出行时间点所需要的时间。 
           
    参数:    
    origin (list): 起始地点的经纬度，经度在前，纬度在后。    
    destination (list): 终点的经纬度，经度在前，纬度在后。       
    transport (str)：交通方式类型，可选参数范围为开车，公共交通。
    time_point (str)：出行时间点，可选参数范围为高峰期和非高峰期。

    返回:    
    int: 两个地点之间指定交通方式在指定出行时间点所需要的时间（单位：分钟）。    
    """ 

    if transport in ["开车", "公共交通"] and time_point in ["高峰期", "非高峰期"]:
        sql = f'''
        SELECT "{transport}" FROM "{time_point}出行时间表" 
        WHERE "起点经纬度" = '{origin[0]},{origin[1]}' AND 
        "终点经纬度" = '{destination[0]},{destination[1]}';
        '''
    else:
        return None
    
    try:
        executor = PostgresQueryExecutor()
        _, time = executor.execute_sql(sql)
        executor.close()
        return time[0][0]
    except:
        return None

In [ ]:
API_agent = create_react_agent(
    model=llm,
    #tools=[rewrite, get_distance, driving_route],
    tools=[get_distance, get_poi, get_time, get_rush_hours],
    prompt=(
        '''
        你是一个负责调用地图查询相关API接口的代理，你的主要任务是负责相关工具的调用并返回工具调用后的结果
        以下是你工作的主要内容：
        - 你只需要回答你所拥有的工具可以解决的相关的问题，其他无关的问题无需回答
        - 你需要分析需要调用的工具所接受的参数类型，传入正确格式的参数
        - 根据工具所能够接受的参数类型和数量，有些问题你可能需要对问题进行分解，多次调用
        - 你需要按照用户的要求对答案进行规范化输出，输出内容中不要包含无关的内容
        - 在回答完问题后，需要将答案交给中枢系统，你无需做任何处理
        '''
    ),
    name="API_agent",
)

In [ ]:
from typing import Annotated
from langchain_core.tools import tool, InjectedToolCallId
from langgraph.prebuilt import InjectedState
from langgraph.graph import StateGraph, START, MessagesState, END
from langgraph.types import Command

def create_handoff_tool(*, agent_name: str, description: str | None = None):
    name = f"transfer_to_{agent_name}"
    description = description or f"Ask {agent_name} for help."

    @tool(name, description=description)
    def handoff_tool(
        state: Annotated[MessagesState, InjectedState],
        tool_call_id: Annotated[str, InjectedToolCallId],
    ) -> Command:
        tool_message = {
            "role": "tool",
            "content": f"Successfully transferred to {agent_name}",
            "name": name,
            "tool_call_id": tool_call_id,
        }
        return Command(
            goto=agent_name,  
            update={**state, "messages": state["messages"] + [tool_message]},  
            graph=Command.PARENT,  
        )

    return handoff_tool


# Handoffs
assign_to_Database_agent = create_handoff_tool(
    agent_name="Database_agent",
    description="Assign task to a Database agent.",
)

assign_to_API_agent = create_handoff_tool(
    agent_name="API_agent",
    description="Assign task to a API agent.",
)

In [ ]:
system_prompt = (
    '''
        1. 你是一个管理两个不同功能代理的管理者，你管理的代理为：API_agent和Database_agent，每个代理的名称和功能的说明如下：
        - API_agent：负责结合地点的经纬度以及其他相关参数，调用地图API来回答距离，时间，周边POI，以及高峰期和非高峰期出行时间相关的问题
        - Database_agent：负责生成SQL语句查询数据库来找到某个地点的经纬度，或回答某POI周边房源，某小区周边其他房源相关的问题
        2. 你每次只能将任务派给一个代理来完成，不要同时将任务派个两个代理，每一个代理会回复它们执行任务后得到的结果
        3. 如果根据某个代理的返回结果你可以回答该问题时，你无需继续分配任务，当你仅获得某个或某些地点的经纬度坐标时，你需要继续分配任务，你无需对这些坐标做过多的分析和推理
        4. 当任务移交给API调用代理时，你需要检查移交内容中是否有地点的经纬度信息，当不存在经纬度信息时，你需要重新决定任务的分配或报告无法回答问题
        5. 不是所有的任务都需要用到两个代理，你需要根据回复决定任务是否结束，如果结束，或者不需要其他代理，你需要将问题最简洁的答案回复给用户
        6. 如果任务没有结束，你需要将上一个代理的回复和用户的问题一起派给下一个代理
        7. 你只需要结合每个代理的回复内容和用户的问题，进行任务的分配和最终回复的输出
        8. 对于数值类的问题，当答案为小数时，你需要以四舍五入的形式保留小数点后2位小数
'''
)

supervisor_agent = create_react_agent(
    model=llm,
    tools=[assign_to_Database_agent, assign_to_API_agent],
    prompt=system_prompt,
    name="supervisor",
)

In [ ]:
with open("./database_agent_prompts/SQL_prompt.json", "r", encoding="utf-8") as file:
    database_prompt = json.load(file)

In [ ]:
with open("./API_agent_prompts/API_prompt.json", "r", encoding="utf-8") as file:
    API_prompt = json.load(file)

In [ ]:
import re
def extract_locations(sql: str) -> list[str]:
    
    value_block = re.compile(
        r'(?:"小区名称"|"POI名称")\s*(?:=|in)\s*('
        r"'[^']+'"           
        r'|'                
        r'\([^)]*\)'         
        r')',
        re.I
    )

    locations = []
    for block in value_block.findall(sql):
        locations.extend(re.findall(r"'([^']+)'", block))
    return locations

In [ ]:
from langchain_core.messages import HumanMessage
import re
from typing import Literal

def API_node(state: MessagesState) -> Command[Literal["supervisor"]]:
    msgs = state["messages"]
    #print(msgs)
    query = re.findall(r'(?<=问题：).*?(?=\n)', msgs[0].content)[-1]
    #print(query)
    intent = re.findall(r'(?<=Intent：).*?(?=\n)', msgs[0].content)[-1]
    #print(intent)
    slots = re.findall(r'(?<=Slots：).*?(?=\n)', msgs[0].content)[-1]
    #print(slots)
    flag = False
    for msg in msgs:
        #print(msg)
        if "<position>" in msg.content:
             pattern = r'<position>(.*?)</position>'
             positions = re.findall(pattern, msg.content)[0]
             flag = True
             break
    if flag == False:
         return_message = "error"
    else:
        prompt = API_prompt[intent].replace("{question}", query).replace("{positions}", str(positions)).replace("{intent}", intent).replace("{slots}", str(slots))
        #print(prompt)
        response = API_agent.invoke({"messages": [{"role": "user", "content": prompt}]})["messages"][-1].content
        pattern = r'<answer>(.*?)</answer>'
        return_message = re.findall(pattern, response)[0]
        return_message = "<answer>" + return_message + "</answer>"

    return Command(
        update={
            "messages": [
                HumanMessage(content=return_message, name="API_agent")
            ]
        },
        goto="supervisor",
    )

def Database_node(state: MessagesState) -> Command[Literal["supervisor"]]:
    state = state["messages"][0].content
    query = re.findall(r'(?<=问题：).*?(?=\n)', state)[-1]
    intent = re.findall(r'(?<=Intent：).*?(?=\n)', state)[-1]
    slots = re.findall(r'(?<=Slots：).*?(?=\n)', state)[-1]
    table_name = get_table_name(query,intent,slots)
    prompt = database_prompt[intent].format(question=query, table_name=table_name, intent=intent, slots=slots)
    #print(prompt)
    response = Database_agent.invoke({"messages": [{"role": "user", "content": prompt}]})["messages"][-1].content
    #print(response)
    pattern = r'<sql>(.*?)</sql>'
    sql = re.findall(pattern, response)[0]
    executor = PostgresQueryExecutor()
    try:
        _, sql_results = executor.execute_sql(sql)
        if intent == "通勤时间查询":
            pos = []
            for result in sql_results:
                  pos.append(str({result[0]:[result[1],result[2]]}))
            return_message = "<position>" + ",".join(pos) + "</position>"
        elif "中心点经度" in sql and "中心点纬度" in sql:
            name_list = extract_locations(sql)
            return_message = "<position>" + str(dict(zip(name_list, sql_results))) + "</position>"
        else:
            return_message = "<answer>" + str(sql_results) + "</answer>"
    except:
            return_message = 'error'
    executor.close() 
    return Command(
        update={
            "messages": [
                HumanMessage(content=return_message, name="Database_agent")
            ]
        },
        goto="supervisor",
    )


builder = StateGraph(MessagesState)
builder.add_edge(START, "supervisor")
builder.add_edge("Database_agent", "supervisor")
builder.add_edge("API_agent", "supervisor")
# builder.add_node("supervisor", supervisor_node)
builder.add_node(supervisor_agent, destinations=("Database_agent", "API_agent", END))
builder.add_node("Database_agent", Database_node)
builder.add_node("API_agent", API_node)
graph = builder.compile()


In [ ]:
with open("./supervisor_agent_prompts/supervisor_prompt.json", "r", encoding="utf-8") as file:
    supervisor_prompt = json.load(file)

In [ ]:
from langchain_core.messages import convert_to_messages


def pretty_print_message(message, indent=False):
    pretty_message = message.pretty_repr(html=True)
    if not indent:
        print(pretty_message)
        return

    indented = "\n".join("\t" + c for c in pretty_message.split("\n"))
    print(indented)


def pretty_print_messages(update, last_message=False):
    is_subgraph = False
    if isinstance(update, tuple):
        ns, update = update
        # skip parent graph updates in the printouts
        if len(ns) == 0:
            return

        graph_id = ns[-1].split(":")[0]
        print(f"Update from subgraph {graph_id}:")
        print("\n")
        is_subgraph = True

    for node_name, node_update in update.items():
        update_label = f"Update from node {node_name}:"
        if is_subgraph:
            update_label = "\t" + update_label

        print(update_label)
        print("\n")

        messages = convert_to_messages(node_update["messages"])
        if last_message:
            messages = messages[-1:]

        for m in messages:
            pretty_print_message(m, indent=is_subgraph)
        print("\n")

In [ ]:
from tqdm import tqdm
import re

function_name_correct = 0
function_parameters_correct = 0
total = 0
#json_file = './test.json'
json_file = '../results/test-with-ICL_pred_intent+slots.json'
with open(json_file, 'r', encoding='utf-8') as f:
    datas = json.load(f)
for data in tqdm(datas):
    query = data['query']
    function_name = data['tool_API']
    intent = data['ICL_pred_intent']
    flag = True
    for key in templates.keys():
        if "+" not in intent and key == intent:
            flag = False
        elif "+" in intent:
            intent_set = set(intent.split("+"))
            key_set = set(key.split("+"))
            if intent_set == key_set:
                intent = key
                flag = False
    intent = "default" if flag else intent
    #print(intent)
    slots = data['ICL_pred_slots']
    chunks = []
    #print(supervisor_prompt[intent].replace("{question}", query).replace("{intent}", intent).replace("{slots}", str(slots)))
    try:
        for chunk in graph.stream(
                {"messages": [{"role": "user", "content":supervisor_prompt[intent].replace("{question}", query).replace("{intent}", intent).replace("{slots}", str(slots))}]}):
            chunks.append(chunk)
            #pretty_print_messages(chunk)
            #print(chunk)
        response = chunks[-1]['supervisor']['messages'][-1].content
        if "<answer>" in response and "</answer>" in response:
            pattern = r'<answer>(.*?)</answer>'
            agent_answer = re.findall(pattern, response)[0]
        else:
            agent_answer = response
        #print(agent_answer)
        data["agent_answer"] = agent_answer
    except:
        data["agent_answer"] = "error"

In [ ]:
save_json_file = f'./results/supervisor_agent-ICL-({model_name.replace(":","")}).json'
with open(save_json_file, 'w', encoding='utf-8') as file:
    json.dump(datas, file, ensure_ascii=False, indent=4)

In [ ]:
import ast

json_file = f'./supervisor_agent-ICL-({model_name.replace(":","")}).json'
count_dict = {# The first element is total number and the second number is correct number
        "Type 1":[0, 0],
        "Type 2":[0, 0],
        "Type 3":[0, 0]
}

with open(json_file, 'r', encoding='utf-8') as f:
    datas = json.load(f)

for data in tqdm(datas):
    if "agent_answer" not in data:
        continue
    function_name = data['tool_API']
    function_par = data['API_parameter']
    QA_type = data["type"]
    true_ans = data['answer']
    agent_ans = re.sub(r'<think>.*?</think>', '', data["agent_answer"], flags=re.S)
    count_dict[QA_type][0] += 1
    if agent_ans == "error" or "无法回答" in agent_ans  or "{" in agent_ans or "}" in agent_ans or "#" in agent_ans or len(agent_ans) > 100:
            continue
    if type(true_ans) == list:
        try:
            agent_ans = ast.literal_eval(agent_ans)
            if len(agent_ans) != len(true_ans):
                continue
            count_dict[QA_type][1] += 1 if set(true_ans) == set(agent_ans) else 0
        except:
            continue
    elif type(true_ans) == float:
        try:
            agent_ans = float(agent_ans)
            count_dict[QA_type][1] += 1 if true_ans == agent_ans else 0
        except:
            continue
    else:
        count_dict[QA_type][1] += 1 if str(true_ans) == agent_ans else 0

In [ ]:
print(f"type1 accuarcy = {count_dict["Type 1"][1]} / {count_dict["Type 1"][0]} = {count_dict["Type 1"][1]/count_dict["Type 2"][0]}")
print(f'type2 accuarcy = {count_dict["Type 2"][1]} / {count_dict["Type 2"][0]} = {count_dict["Type 2"][1]/count_dict["Type 2"][0]}')
print(f'type3 accuarcy = {count_dict["Type 3"][1]} / {count_dict["Type 3"][0]} = {count_dict["Type 3"][1]/count_dict["Type 3"][0]}')
print(f"overall accuarcy = {count_dict["Type 1"][1] + count_dict["Type 2"][1] + count_dict["Type 3"][1]} / {count_dict["Type 1"][1] + count_dict["Type 1"][1] + count_dict["Type 1"][1]} = {(count_dict["Type 1"][1] + count_dict["Type 2"][1] + count_dict["Type 3"][1])/(count_dict["Type 1"][0] + count_dict["Type 2"][0] + count_dict["Type 3"][0])}")

In [ ]:
import ast

json_file = f'./supervisor_agent-bert-({model_name.replace(":","")}).json'
count_dict = {# The first element is total number and the second number is F1 score
        "Type 1":[0, 0],
        "Type 2":[0, 0],
        "Type 3":[0, 0]
}

with open(json_file, 'r', encoding='utf-8') as f:
    datas = json.load(f)

for data in tqdm(datas):
    if "agent_answer" not in data:
        continue
    function_name = data['function_name']
    function_par = data['function_parameter']
    QA_type = data["type"]
    true_ans = data['answer']
    agent_ans = re.sub(r'<think>.*?</think>', '', data["agent_answer"], flags=re.S)
    count_dict[QA_type][0] += 1
    if agent_ans == "error" or "无法回答" in agent_ans  or "{" in agent_ans or "}" in agent_ans or "#" in agent_ans or len(agent_ans) > 100:
            continue
    if type(true_ans) == list:
        gt = len(true_ans)
        try:
            agent_ans = ast.literal_eval(agent_ans)
            pr = len(agent_ans)
            correct = 0
            for ans in true_ans:
                if ans in agent_ans:
                    correct += 1
                p = correct / pr
                r = correct / gt
                count_dict[QA_type][1] += 2 * p * r / (p + r)
        except:
            continue
    elif type(true_ans) == float:
        try:
            agent_ans = float(agent_ans)
            count_dict[QA_type][1] += 1 if true_ans == agent_ans else 0
        except:
            continue
    else:
        count_dict[QA_type][1] += 1 if str(true_ans) == agent_ans else 0

In [ ]:
print(f"type1 F1 = {count_dict["Type 1"][1]} / {count_dict["Type 1"][0]} = {count_dict["Type 1"][1]/count_dict["Type 2"][0]}")
print(f'type2 F1 = {count_dict["Type 2"][1]} / {count_dict["Type 2"][0]} = {count_dict["Type 2"][1]/count_dict["Type 2"][0]}')
print(f'type3 F1 = {count_dict["Type 3"][1]} / {count_dict["Type 3"][0]} = {count_dict["Type 3"][1]/count_dict["Type 3"][0]}')
print(f"overall F1 = {count_dict["Type 1"][1] + count_dict["Type 2"][1] + count_dict["Type 3"][1]} / {count_dict["Type 1"][0] + count_dict["Type 1"][0] + count_dict["Type 1"][0]} = {(count_dict["Type 1"][1] + count_dict["Type 2"][1] + count_dict["Type 3"][1])/(count_dict["Type 1"][0] + count_dict["Type 2"][0] + count_dict["Type 3"][0])}")